# 02 PID Control Analysis

Compare manual, relay-based Ziegler-Nichols and numerical grid-search tuning on the same motor model.

In [ ]:
# Colab bootstrap: install only free packages and load this repository.
!pip -q install numpy pandas matplotlib scipy
from pathlib import Path
import subprocess, sys
REPO_URL = 'https://github.com/SaadWajih99/industrial-conveyor-plc-hmi'
PROJECT = Path('industrial-conveyor-plc-hmi')
if not (PROJECT / 'simulation').exists():
    if 'REPLACE_WITH_GITHUB_USERNAME' in REPO_URL:
        raise RuntimeError('Set REPO_URL to the public GitHub repository URL before running this notebook.')
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT.resolve()))


In [ ]:
import pandas as pd
from control.pid_tuning import MotorModel, manual_tuning, ziegler_nichols_tuning, optimize_tuning, simulate_controller
tunings = {'Manual': manual_tuning(), 'Ziegler-Nichols': ziegler_nichols_tuning(MotorModel()), 'Grid search': optimize_tuning()}
rows = []
for method, gains in tunings.items():
    result = simulate_controller(gains, measurement_noise_std=0.005)
    rows.append({'method': method, **result['metrics']})
    pd.Series(result['metrics']).plot.bar(title=method)
print(pd.DataFrame(rows).to_string(index=False))


The model includes output saturation, conditional anti-windup/back-calculation and a first-order derivative filter. Metrics are calculated from the generated response arrays.